# Data Analysis
## Planck 2018 + WMAP 9-year CMB TT Power Spectra

Datasets acquired via PTorrent from:
- Planck Legacy Archive (PLA) — `COM_PowerSpect_CMB-TT-full_R3.01.txt`
- NASA LAMBDA — `wmap_tt_spectrum_9yr_v5.txt`

Three experiments run in sequence:
1. `fractal_boundary.py` — DFA, local spectral index, opposition signature
2. `ptychography.py` — ePIE reconstruction + fractal extrapolation to ℓ=5450
3. `ask_boundary.py` — direct interrogation of ℓ* values

Source: `Ainulindale/Experiments/FractalBoundary/`

In [ ]:
import numpy as np
import os

DATA    = "data"
RESULTS = "results"

# ── load spectra ─────────────────────────────────────────────────────────────
def load(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            parts = line.split()
            if len(parts) >= 2:
                try: rows.append([float(p) for p in parts[:2]])
                except: pass
    a = np.array(rows)
    ell = a[:,0]; Dl = a[:,1]
    Cl  = Dl * 2*np.pi / (ell*(ell+1))
    return ell, Cl, Dl

ell_p, Cl_p, Dl_p = load(os.path.join(DATA, 'planck_tt_2018.txt'))
ell_w, Cl_w, Dl_w = load(os.path.join(DATA, 'wmap_tt_9yr.txt'))

print(f"Planck 2018 : {len(ell_p)} bins  ℓ={ell_p[0]:.0f}..{ell_p[-1]:.0f}")
print(f"WMAP 9yr    : {len(ell_w)} bins  ℓ={ell_w[0]:.0f}..{ell_w[-1]:.0f}")

In [ ]:
# ── load ptychographic reconstruction ────────────────────────────────────────
ell_recon = np.load(os.path.join(RESULTS, 'ell_reconstruction.npy'))
Cl_recon  = np.load(os.path.join(RESULTS, 'Cl_reconstruction.npy'))
ell_ext   = np.load(os.path.join(RESULTS, 'ell_extrapolation.npy'))
Cl_ext    = np.load(os.path.join(RESULTS, 'Cl_extrapolation.npy'))

print(f"Reconstruction : ℓ={ell_recon[0]:.0f}..{ell_recon[-1]:.0f}  ({len(ell_recon)} bins)")
print(f"Extrapolation  : ℓ={ell_ext[0]:.0f}..{ell_ext[-1]:.0f}  ({len(ell_ext)} bins)")

In [ ]:
# ── compare predictions to measurements ──────────────────────────────────────
PHI = (1 + np.sqrt(5)) / 2

# from fractal_boundary.py results
Df_planck  = 0.458637
Df_wmap    = 0.759565
Df_mean    = 0.609101
ell_star_p = 2450.0
ell_star_w = 1179.0

# from ask_boundary.py results
Cl_star_planck = 2.440394e-04
Cl_star_wmap   = -1.605311e-03   # NEGATIVE — zero divisor confirmed
ell_ratio      = 2450 / 1179
opposition_ell = 2.0             # quadrupole
epie_iters     = 31

print("PREDICTION vs MEASUREMENT")
print("=" * 56)
print(f"P1  D_f = φ−1 = {PHI-1:.4f}    measured = {Df_mean:.4f}    "
      f"err = {abs(Df_mean-(PHI-1)):.4f}  {'✓' if abs(Df_mean-(PHI-1)) < 0.02 else '?'}")
print(f"P2  ℓ*_P/ℓ*_W = 2.000      measured = {ell_ratio:.4f}    "
      f"err = {abs(ell_ratio-2):.4f}  {'✓' if abs(ell_ratio-2) < 0.1 else '?'}")
print(f"P3  C_ℓ*(WMAP) < 0         measured = {Cl_star_wmap:.3e}   {'✓' if Cl_star_wmap < 0 else '✗'}")
print(f"P4  opposition at ℓ=2      measured = ℓ={opposition_ell:.0f}   {'✓' if opposition_ell == 2 else '✗'}")
print(f"P5  ePIE < 50 iter         measured = {epie_iters}   {'✓' if epie_iters < 50 else '✗'}")